The goal of this penultimate notebook is to choose the best model for data derived in nb7. It deals exclusively with model architecture and fine tuning.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import xgboost as xgb
import optuna
import time

from helpers import submissions, final_model_helper

from helpers.utils import sklearn_helper, optuna_helper

from sklearn.pipeline import Pipeline
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import VotingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score, roc_auc_score, f1_score

from imblearn.over_sampling import SMOTE


from lightgbm import LGBMClassifier

In [2]:
data = joblib.load("data/processed/all_tables.jbl")
data.info()

X_train = data[~data["TARGET"].isna()]
X_train = X_train.drop(columns=["SK_ID_CURR"]).reset_index(drop=True)
y_train = X_train.pop("TARGET")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 356255 entries, 0 to 356254
Columns: 749 entries, SK_ID_CURR to prev_app_onehot_count_NAME_CONTRACT_STATUS_S_y
dtypes: bool(34), category(14), float64(694), int64(6), object(1)
memory usage: 1.9+ GB


We use an adapter that simplifies cross validating models.

In [3]:
def val_model(model, X_train=X_train, y_train=y_train) -> pd.DataFrame:
    """5CV model"""
    return sklearn_helper.stratified_cv_model(
        model, X_train, y_train, scoring=["average_precision", "roc_auc", "f1_macro"]
    )

# Dedicated model libraries
## LightGBM

#### Gradient Boosted Tree

In [4]:
val_model(LGBMClassifier(boosting_type="gbdt", verbose=-1))

,average_precision,roc_auc,f1_macro
mean,0.2710,0.7800,0.5166
std,0.0072,0.0038,0.0009


#### Random_forest

In [5]:
val_model(
    LGBMClassifier(boosting_type="rf", verbose=-1, bagging_fraction=0.2, bagging_freq=5)
)

,average_precision,roc_auc,f1_macro
mean,0.2113,0.7214,0.5552
std,0.0051,0.0041,0.0025


The parameters used, make the RF significantly underperform when compared to gradient boosted trees.
* F1 macro score is improved only due to class imbalance.

#### DART

In [6]:
val_model(LGBMClassifier(boosting_type="dart", verbose=-1))

,average_precision,roc_auc,f1_macro
mean,0.2584,0.7680,0.4864
std,0.0071,0.0039,0.0005


Performance slightly worse in all regards, when compared to gradient boosted trees.

## XGBoost
XGboost provides alternative gradient boosted tree application. More importantly, it has been shown to outperform LightGBM for large datasets.

In [7]:
model = xgb.XGBClassifier(
    objective="binary:logistic",
    enable_categorical=True,
)

val_model(model)

,average_precision,roc_auc,f1_macro
mean,0.2496,0.7663,0.5376
std,0.0050,0.0030,0.0016


# Nan-sensitive models
Models unable to impute nan values will need to rely on an imputer. 

## NAN classification
As data imputation will be introducing noise and bias, it is important to first quantify how much data is missing and whether features need to be discarded.

Models of different architecture are tested with the goal of building an ensemble. 

### Preprocessor
For further work integration the categorical columns are saved and loaded into an accessible file.

In [4]:
num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
num_cols = [col for col in num_cols if X_train[col].notna().sum() > 100]


cat_cols = X_train.select_dtypes(include=["category"]).columns.tolist()
cat_cols = [col for col in cat_cols if X_train[col].notna().sum() > 100]

nessesary_columns = {
    "X_cols": X_train.columns,
    "y_col": "TARGET",
    "id_cols": ["SK_ID_CURR", "SK_ID_BUREAU", "SK_ID_PREV"],
    "num_cols": X_train.select_dtypes(include=[np.number]).columns.tolist(),
    "cat_cols": X_train.select_dtypes(include=["category"]).columns.tolist(),
    "bool_cols": X_train.select_dtypes(include=["bool"]).columns.tolist(),
}

print("Number of columns:", len(nessesary_columns["X_cols"]))

joblib.dump(nessesary_columns, "models/model_columns.jbl")

Number of columns: 747


['models/model_columns.jbl']

These are then used to build a preprocessor:
* Missing numerics are imputed as median.
* Missing categorical values are imputed as most frequent
* Categorical data is encoded as str for SKLEARN compatability

In [5]:
preprocessor = final_model_helper.preprocessor_for_sklearn(
    columns_path="models/model_columns.jbl"
).get_preprocessor()

display(preprocessor)

Loaded X and y columns: 747 + 1


ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['CNT_CHILDREN', 'AMT_INCOME_TOTAL',
                                  'AMT_CREDIT', 'AMT_ANNUITY',
                                  'AMT_GOODS_PRICE',
                                  'REGION_POPULATION_RELATIVE', 'DAYS_BIRTH',
                                  'DAYS_EMPLOYED', 'DAYS_REGISTRATION',
                                  'DAYS_ID_PUBLISH', 'OWN_CAR_AGE',
                                  'CNT_FAM_MEMBERS', 'REGION_RATI...
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('onehot',
                                                  OneHotEncoder(handle_unknown='ignore'))]),
                                 ['NAME_CONTRACT_TYPE', 'CODE_GENDER',
                                  'NAME_TYPE_SUITE', 'NAME_INCOME_TYPE',
                                  'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS',
                                  'NAME_HOUSING_TYPE', 'OCCUPATION_TYPE',
                                  'WEEKDAY_APPR_PROCESS_START',
                                  'ORGANIZATION_TYPE', 'FONDKAPREMONT_MODE',
                                  'HOUSETYPE_MODE', 'WALLSMATERIAL_MODE',
                                  'EMERGENCYSTATE_MODE'])])

In [ ]:
X_imputed = preprocessor.fit_transform(X_train)
X_imputed.shape

This preprocessor is then tested on 4 unique structure sklearn models capable of outputting ROC_AUC curves.

In [10]:
sk_models = [
    LogisticRegression(max_iter=1000, random_state=3),
    RandomForestClassifier(random_state=3),
    KNeighborsClassifier(),
    HistGradientBoostingClassifier(random_state=3),
]


def create_pipes(model, preprocessor):
    pipe = Pipeline([("preprocessor", preprocessor), (type(model).__name__, model)])
    return pipe

In [11]:
for model in sk_models:
    start_time = time.time()

    pipe = create_pipes(model, preprocessor)

    print(f"Validating: {model}")
    display(val_model(pipe, X_train))
    end_time = time.time()

    execution_time = end_time - start_time
    print(f"\nDuration: {execution_time:.2f} seconds")

Validating: LogisticRegression(max_iter=1000, random_state=3)


c:\GITHUB\Turing submissions\psauci-DS.v2.5.3.4.1\.venv\Lib\site-packages\sklearn\impute\_base.py:635: UserWarning: Skipping features without any observed values: ['std_credit_usage_Other']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
c:\GITHUB\Turing submissions\psauci-DS.v2.5.3.4.1\.venv\Lib\site-packages\sklearn\impute\_base.py:635: UserWarning: Skipping features without any observed values: ['std_credit_usage_Other']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
c:\GITHUB\Turing submissions\psauci-DS.v2.5.3.4.1\.venv\Lib\site-packages\sklearn\impute\_base.py:635: UserWarning: Skipping features without any observed values: ['std_credit_usage_Other']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


,average_precision,roc_auc,f1_macro
mean,0.2501,0.7683,0.5102
std,0.0053,0.0038,0.0017



Duration: 451.43 seconds
Validating: RandomForestClassifier(random_state=3)


c:\GITHUB\Turing submissions\psauci-DS.v2.5.3.4.1\.venv\Lib\site-packages\sklearn\impute\_base.py:635: UserWarning: Skipping features without any observed values: ['std_credit_usage_Other']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
c:\GITHUB\Turing submissions\psauci-DS.v2.5.3.4.1\.venv\Lib\site-packages\sklearn\impute\_base.py:635: UserWarning: Skipping features without any observed values: ['std_credit_usage_Other']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
c:\GITHUB\Turing submissions\psauci-DS.v2.5.3.4.1\.venv\Lib\site-packages\sklearn\impute\_base.py:635: UserWarning: Skipping features without any observed values: ['std_credit_usage_Other']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


...

,average_precision,roc_auc,f1_macro
mean,0.1019,0.5729,0.4995
std,0.0006,0.0018,0.0027



Duration: 3124.44 seconds
Validating: HistGradientBoostingClassifier(random_state=3)


c:\GITHUB\Turing submissions\psauci-DS.v2.5.3.4.1\.venv\Lib\site-packages\sklearn\impute\_base.py:635: UserWarning: Skipping features without any observed values: ['std_credit_usage_Other']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
c:\GITHUB\Turing submissions\psauci-DS.v2.5.3.4.1\.venv\Lib\site-packages\sklearn\impute\_base.py:635: UserWarning: Skipping features without any observed values: ['std_credit_usage_Other']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
c:\GITHUB\Turing submissions\psauci-DS.v2.5.3.4.1\.venv\Lib\site-packages\sklearn\impute\_base.py:635: UserWarning: Skipping features without any observed values: ['std_credit_usage_Other']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


,average_precision,roc_auc,f1_macro
mean,0.2693,0.7780,0.5153
std,0.0082,0.0042,0.0019



Duration: 478.81 seconds


* The performance of HistGradientBoostingClassifier is near identical to that of LGBM GBDT. Given that both algorithms are rather similar of implementations of decisions trees, they can be treated as near interchangeable. lgbm will be used due to performance gains on large data.
* LogisticRegression provides comparable performance and trains at acceptable rate.
* KNN performs very poorly due to imbalance in the dataset.
* Random Forest performs comparably to LGBM implementation, but at a much slower training time.

In conclusion, the data imputation and preprocessing techniques have minimal effect on models performance and can be considered appropriate.

#### Smote and KNN
To account for data imbalance and better evaluate kNN performance, SMOTE will be used to generate extra samples.
Due to knn being a slow model the need to handle generated data, CV will not be used

In [12]:
X_imp_train, X_imp_test, y_imp_train, y_imp_test = train_test_split(
    X_imputed, y_train, test_size=0.2, random_state=42, stratify=y_train
)

smote = SMOTE(random_state=5)

X_smote, y_smote = smote.fit_resample(X_imp_train, y_imp_train)
print("Smote generate X shape:", X_smote.shape)
print("From orginal X shape:", X_imp_train.shape)

model = KNeighborsClassifier()

model.fit(X_smote, y_smote)

y_pred = model.predict(X_imp_test)

print(f"Average Precision: {average_precision_score(y_imp_test, y_pred):.4f}")
print(f"ROC AUC: {roc_auc_score(y_imp_test, y_pred):.4f}")
print(f"F1 Macro: {f1_score(y_imp_test, y_pred, average='macro'):.4f}")

Smote generate X shape: (452296, 841)
From orginal X shape: (246008, 841)
Average Precision: 0.0921
ROC AUC: 0.5638
F1 Macro: 0.3734


Imputing positive class labels to account for imbalanced data, did not improve KNN performance to a desirable level.

# Fine Tuning
Given the large dataset size and small STD of metrics, we do not need to use CV for fine tuning

### GBDT fine tuning
This is not run fully due to time and computation constrains.

In [13]:
def gbdt_params(trial: optuna.trial.Trial) -> dict:
    return {
        "is_unbalance": True,
        "verbosity": -1,
        "boosting_type": "gbdt",
        "n_estimators": trial.suggest_int("n_estimators", 100, 2000),
        "stop_round": 50,
        "lambda_l1": trial.suggest_float("lambda_l1", 1e-8, 10.0, log=True),
        "lambda_l2": trial.suggest_float("lambda_l2", 1e-8, 10.0, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 2, 256),
        "feature_fraction": trial.suggest_float("feature_fraction", 0.4, 1.0),
        "bagging_fraction": trial.suggest_float("bagging_fraction", 0.4, 1.0),
        "bagging_freq": trial.suggest_int("bagging_freq", 1, 7),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 100),
    }


study_results = lambda: optuna_helper.run_study(
    LGBMClassifier(random_state=326), gbdt_params, X_train, y_train, n_trials=100
)
gbdt_result = optuna_helper.load_or_create_study("gbdt_study_results", study_results)
gbdt_result.best_params

[I 2025-08-07 23:36:08,241] A new study created in memory with name: no-name-c88ffcec-f0c4-43f3-b222-f450af436d8f
[W 2025-08-07 23:39:06,615] Trial 0 failed with parameters: {'n_estimators': 1306, 'lambda_l1': 3.781405799592358e-06, 'lambda_l2': 0.0013583459167249576, 'num_leaves': 234, 'feature_fraction': 0.9387650919770139, 'bagging_fraction': 0.4862518997839145, 'bagging_freq': 2, 'min_child_samples': 58} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "c:\GITHUB\Turing submissions\psauci-DS.v2.5.3.4.1\.venv\Lib\site-packages\optuna\study\_optimize.py", line 201, in _run_trial
    value_or_values = func(trial)
  File "c:\GITHUB\Turing submissions\psauci-DS.v2.5.3.4.1\helpers\utils\optuna_helper.py", line 75, in objective
    return objective_function(trial, pipeline, params, X_train, y_train, scoring_function)
  File "c:\GITHUB\Turing submissions\psauci-DS.v2.5.3.4.1\helpers\utils\optuna_helper.py", line 52, in objective_function
    mo

KeyboardInterrupt: 

### Logistic Regression Tuning

In [ ]:
def logreg_params(trial: optuna.Trial) -> dict:
    """Suggest parameters for Logistic Regression model based on solver used"""
    return {
        "penalty": trial.suggest_categorical("penalty", ["elasticnet"]),
        "C": trial.suggest_float("C", 1e-5, 100, log=True),
        "solver": trial.suggest_categorical("solver", ["saga"]),
        "max_iter": trial.suggest_int("max_iter", 500, 2000),
        "l1_ratio": trial.suggest_float("l1_ratio", 0.0, 1.0),
    }


study_results = lambda: optuna_helper.run_study(
    LogisticRegression(random_state=326, max_iter=1000),
    logreg_params,
    X_imputed,
    y_train,
    n_trials=2,
)
logreg_result = optuna_helper.load_or_create_study("logreg_results", study_results)
logreg_result.best_params

[I 2025-08-08 00:03:29,020] A new study created in memory with name: no-name-ecf9144e-836f-4963-a322-d38afe8d8f39


# Voting Classifier
The best performing models are then investigated along their ROC and AUC curves.

In [6]:
class_sk_models = [LogisticRegression(max_iter=1000, random_state=3)]
voter = VotingClassifier(
    estimators=[
        ("lgbm_gbdt", LGBMClassifier(boosting_type="gbdt", verbose=-1)),
    ]
    + [
        (
            type(model).__name__,
            Pipeline([("preprocessor", preprocessor), (type(model).__name__, model)]),
        )
        for model in class_sk_models
    ],
    voting="soft",
    n_jobs=-1,
)


display(voter)

VotingClassifier(estimators=[('lgbm_gbdt', LGBMClassifier(verbose=-1)),
                             ('LogisticRegression',
                              Pipeline(steps=[('preprocessor',
                                               ColumnTransformer(transformers=[('num',
                                                                                Pipeline(steps=[('imputer',
                                                                                                 SimpleImputer(strategy='median')),
                                                                                                ('scaler',
                                                                                                 StandardScaler())]),
                                                                                ['CNT_CHILDREN',
                                                                                 'AMT_INCOME_TOTAL',
                                                                                 'AMT_CREDIT',
                                                                                 'AMT_ANNUITY',
                                                                                 'AMT_GOODS_PRICE',
                                                                                 'REGION_POPULATI...
                                                                                 'NAME_TYPE_SUITE',
                                                                                 'NAME_INCOME_TYPE',
                                                                                 'NAME_EDUCATION_TYPE',
                                                                                 'NAME_FAMILY_STATUS',
                                                                                 'NAME_HOUSING_TYPE',
                                                                                 'OCCUPATION_TYPE',
                                                                                 'WEEKDAY_APPR_PROCESS_START',
                                                                                 'ORGANIZATION_TYPE',
                                                                                 'FONDKAPREMONT_MODE',
                                                                                 'HOUSETYPE_MODE',
                                                                                 'WALLSMATERIAL_MODE',
                                                                                 'EMERGENCYSTATE_MODE'])])),
                                              ('LogisticRegression',
                                               LogisticRegression(max_iter=1000,
                                                                  random_state=3))]))],
                 n_jobs=-1, voting='soft')

In [ ]:
val_model(voter)

c:\GITHUB\Turing submissions\psauci-DS.v2.5.3.4.1\.venv\Lib\site-packages\sklearn\impute\_base.py:635: UserWarning: Skipping features without any observed values: ['std_credit_usage_Other']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
c:\GITHUB\Turing submissions\psauci-DS.v2.5.3.4.1\.venv\Lib\site-packages\sklearn\impute\_base.py:635: UserWarning: Skipping features without any observed values: ['std_credit_usage_Other']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


The model outperforms pure lgbm gbdt across all metrics. It will be used as a final model for submission.

## Final Model

In [ ]:
X_test = data[data["TARGET"].isna()]
X_id = X_test.pop("SK_ID_CURR")
voter.fit(X_train, y_train)
submissions.prepare_submission(
    voter,
    X_test.drop(columns=["TARGET"]),
    X_id,
    "voter_log_reg_gbdt",
)

Submission created: submissions\sub_2025-08-06_17-35_voter_log_reg_gbdt.csv


'submissions\\sub_2025-08-06_17-35_voter_log_reg_gbdt.csv'

The achieved public score score is 0.777109 vs 0.76818 (LGBM only implementation). Building of the classifier model is thus deemed success.